# GFS download on Colab

Use a **CPU** runtime (not a GPU) to save compute units. Nothing here needs a GPU.

Before running: upload `balloon_code.zip` (from `dist/` in the project) into the shared `balloon-forecaster` Drive folder.

Run the cells in order. Cell 1 is the speed test; paste its output back to Claude.

## 1. Speed test to AWS (same test that was slow on the laptop)

In [1]:
import time, requests
from concurrent.futures import ThreadPoolExecutor

URL = 'https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs.20260101/12/atmos/gfs.t12z.pgrb2.0p50.f024'

def fetch(start, nbytes):
    r = requests.get(URL, headers={'Range': f'bytes={start}-{start + nbytes - 1}'}, timeout=120)
    return len(r.content)

t = time.time(); n = fetch(0, 30_000_000); dt = time.time() - t
print(f'1 connection : {n/1e6:.0f} MB in {dt:.1f}s = {n/1e6/dt:.1f} MB/s')

t = time.time()
with ThreadPoolExecutor(8) as ex:
    total = sum(ex.map(lambda i: fetch(40_000_000 + i * 5_000_000, 5_000_000), range(8)))
dt = time.time() - t
print(f'8 connections: {total/1e6:.0f} MB in {dt:.1f}s = {total/1e6/dt:.1f} MB/s total')

1 connection : 30 MB in 1.1s = 27.5 MB/s
8 connections: 40 MB in 0.5s = 73.3 MB/s total


## 2. Mount Drive, unpack the code, install packages

In [ ]:
import os, zipfile
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/balloon-forecaster'
CODE_DIR = '/content/code'
assert os.path.isdir(DATA_ROOT), f'{DATA_ROOT} not found: add the shared folder shortcut to My Drive'

bundle = f'{DATA_ROOT}/balloon_code.zip'
assert os.path.exists(bundle), 'Upload balloon_code.zip into the balloon-forecaster Drive folder first'
zipfile.ZipFile(bundle).extractall(CODE_DIR)
print('code unpacked:', sorted(os.listdir(CODE_DIR)))

In [ ]:
# GRIB reader. ecmwflibs bundles the ecCodes library; apt is a fallback.
!pip -q install netCDF4 cfgrib ecmwflibs

import subprocess, sys
check = subprocess.run([sys.executable, '-c', 'import cfgrib, eccodes; print("cfgrib", cfgrib.__version__, "ok")'],
                       capture_output=True, text=True)
print(check.stdout or check.stderr[-800:])

If the check above failed, run this cell, then rerun the check:
```
!apt-get -qq install -y libeccodes-dev && pip -q install eccodes
```

## 3. Pilot download: September 2026 (30 days, all 21 forecast steps)
Measured on Colab: about 20 s per day, so this should take about 10 minutes. It is safe to stop and rerun: finished days are skipped. To change the range, edit START and END below.

In [ ]:
START, END, EVERY = '2026-09-01', '2026-09-30', 1   # pilot month
WORKERS = 4   # forecast steps downloaded at once (each uses 8 connections)

!cd {CODE_DIR} && BALLOON_DATA_DIR={DATA_ROOT} python -u -m scripts.fetch_gfs --start {START} --end {END} --every {EVERY} --workers {WORKERS}

## 4. Verify what landed on Drive

In [ ]:
import glob
import xarray as xr

# Only current-format (v2) files. Open each one fully, which also catches any file damaged
# by a crash at shutdown.
files = sorted(glob.glob(f'{DATA_ROOT}/processed/gfs/gfs_0p50_v2_*.nc'))
bad = []
for f in files:
    try:
        with xr.open_dataset(f) as ds:
            ds.load()
            assert ds.sizes['lead_h'] == 21 and ds.sizes['isobaricInhPa'] == 21
    except Exception as e:
        bad.append((os.path.basename(f), repr(e)))
print(len(files), 'v2 files,', round(sum(os.path.getsize(f) for f in files) / 1e6, 1), 'MB,', len(bad), 'unreadable')
for name, err in bad: print('  BAD', name, err)
for f in files[:5]: print(' ', os.path.basename(f))